# Tool-Call Fine-Tune Lab — QLoRA Pipeline (T4-Safe v3)

Fine-tunes **Qwen2.5-7B-Instruct** using QLoRA on Kaggle T4.

**v3 fixes:** No pip installs (use pre-installed packages), no internet needed, no merge, no eval during training.

In [ ]:
# Cell 0 — Verify pre-installed packages (NO pip install)
import importlib
for mod in ['bitsandbytes', 'peft', 'accelerate', 'transformers', 'datasets', 'torch']:
    m = importlib.import_module(mod)
    print(f"{mod}: {getattr(m, '__version__', 'ok')}")
print('\nAll packages available.')

In [ ]:
# Cell 1 — Config
import os, gc, json, torch
from pathlib import Path

os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    try: HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except: pass
except: pass

DATA_DIR = Path('/kaggle/input/tool-call-finetune-data')
LORA_DIR = Path('/kaggle/working/qwen25-7b-tool-call-lora')
RESULTS_DIR = Path('/kaggle/working/results')
LORA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Find cached Qwen model
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
for p in sorted(Path('/kaggle/input').rglob('config.json')):
    try:
        cfg = json.loads(p.read_text())
        if 'qwen2' in cfg.get('model_type', '').lower():
            BASE_MODEL = str(p.parent)
            print(f'Cached model: {BASE_MODEL}')
            break
    except: continue
else:
    print(f'Will download: {BASE_MODEL}')
    for c in Path('/kaggle/input').iterdir(): print(f'  {c.name}/')

MAX_SEQ_LENGTH = 1024
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32

print(f'Model: {BASE_MODEL}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Cell 2 — Load data
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

train_data = load_jsonl(DATA_DIR / 'train.jsonl')
test_data = load_jsonl(DATA_DIR / 'test.jsonl')
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')

In [ ]:
# Cell 3 — Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none', task_type='CAUSAL_LM',
))
model.print_trainable_parameters()
print(f'VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
# Cell 4 — Format + tokenize
from datasets import Dataset

def fmt(ex):
    msgs, tools = ex.get('messages', []), ex.get('tools', [])
    td = ''
    if tools:
        td = '\n\nAvailable tools:'
        for t in tools:
            fn = t.get('function', {})
            td += f"\n- {fn.get('name','')}: {fn.get('description','')}"
    parts = []
    for m in msgs:
        r, c = m['role'], m.get('content') or ''
        if r == 'system': parts.append(f'<|im_start|>system\n{c}{td}<|im_end|>')
        elif r == 'user': parts.append(f'<|im_start|>user\n{c}<|im_end|>')
        elif r == 'tool': parts.append(f'<|im_start|>tool\n{c}<|im_end|>')
        elif r == 'assistant':
            tcs = m.get('tool_calls', [])
            if tcs:
                ss = []
                for tc in tcs:
                    fn = tc.get('function', {})
                    a = fn.get('arguments', {})
                    if isinstance(a, str):
                        try: a = json.loads(a)
                        except: pass
                    ss.append(json.dumps({'name': fn.get('name',''), 'arguments': a}))
                ac = '<tool_call>\n' + '\n'.join(ss) + '\n</tool_call>'
            else: ac = c
            parts.append(f'<|im_start|>assistant\n{ac}<|im_end|>')
    return {'text': '\n'.join(parts)}

ds = Dataset.from_list(train_data).map(fmt, desc='fmt')
ds = ds.filter(lambda x: len(tokenizer.encode(x['text'])) <= MAX_SEQ_LENGTH, desc='filter')
print(f'After filter: {len(ds):,}')

def tok(examples):
    o = tokenizer(examples['text'], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    o['labels'] = [ids[:] for ids in o['input_ids']]
    return o

train_tok = ds.map(tok, batched=True, remove_columns=ds.column_names, desc='tok')
print(f'Tokenized: {len(train_tok):,}')
del ds, train_data; gc.collect()

In [ ]:
# Cell 5 — Train
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir=str(LORA_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=True, bf16=False,
    logging_steps=50,
    save_strategy='steps', save_steps=500, save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    eval_strategy='no',
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print(f'Steps: {len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)}')
print(f'VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')
print('\n' + '='*50 + '\nTRAINING STARTED\n' + '='*50)

trainer.train()

print('\n' + '='*50 + '\nTRAINING COMPLETE\n' + '='*50)

In [ ]:
# Cell 6 — Save adapter + metrics
trainer.save_model(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))

metrics = trainer.state.log_history
(RESULTS_DIR / 'training_metrics.json').write_text(json.dumps(metrics, indent=2))

losses = [m['loss'] for m in metrics if 'loss' in m]
if losses:
    print(f'Final loss: {losses[-1]:.4f}')
    print(f'Min loss:   {min(losses):.4f}')

print(f'\nSaved to {LORA_DIR}')
for f in sorted(LORA_DIR.iterdir()):
    print(f'  {f.name}: {f.stat().st_size / 1024**2:.1f} MB')

In [ ]:
# Cell 7 — Sanity check
model.eval()
prompt = (
    '<|im_start|>system\nYou are a helpful assistant.\n\n'
    'Available tools:\n- get_weather: Get current weather<|im_end|>\n'
    '<|im_start|>user\nWeather in Tokyo?<|im_end|>\n'
    '<|im_start|>assistant\n'
)
inp = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=150, do_sample=False)
gen = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=False)
print(f'Output: {gen[:300]}')
print(f'Contains get_weather: {"get_weather" in gen}')

In [ ]:
# Cell 8 — Push to HF (optional)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    repo = 'doeonkim00/qwen2.5-7b-tool-calling-lora'
    api.create_repo(repo, exist_ok=True, private=True)
    api.upload_folder(folder_path=str(LORA_DIR), repo_id=repo,
                      commit_message=f'QLoRA adapter — loss {losses[-1]:.4f}' if losses else 'QLoRA adapter')
    print(f'Pushed to https://huggingface.co/{repo}')
else:
    print('No HF_TOKEN — download from Output tab')

print('\nDONE.')